# SPINE-GPE v7 — PNADc Historical Backcast Hardening v1.1.0

Este notebook executa a auditoria e o hardening final do backcast histórico já certificado pelo engine v1.0.1.

**Estimando oficial:** soma ponderada das probabilidades do `mapping_pooled`.

**Limite:** fora dos módulos especiais, plataforma direta permanece não observada.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7')
SCRIPTS = ROOT / 'scripts'
SCRIPT = SCRIPTS / 'SPINE_GPEv7_PNADC_HISTORICAL_BACKCAST_HARDENING_v1.1.0.py'
REQ = SCRIPTS / 'requirements_SPINE_GPEv7_PNADC_HISTORICAL_BACKCAST_HARDENING_v1.1.0.txt'

assert ROOT.exists(), ROOT
assert SCRIPT.exists(), SCRIPT
assert REQ.exists(), REQ
print('ROOT:', ROOT)
print('SCRIPT:', SCRIPT)

In [ ]:
import subprocess, sys

install = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQ)],
    text=True,
    capture_output=True,
    check=False,
)
print(install.stdout)
print(install.stderr)
print('Install exit code:', install.returncode)
assert install.returncode == 0

## 1. Auditoria leve

Valida lock upstream, hashes, parser e layouts antes do ajuste de qualquer modelo.

In [ ]:
audit = subprocess.run(
    [
        sys.executable, str(SCRIPT),
        '--root', str(ROOT),
        '--mode', 'audit',
        '--strict',
    ],
    text=True,
    capture_output=True,
    check=False,
)
print(audit.stdout)
print(audit.stderr)
print('Audit exit code:', audit.returncode)

In [ ]:
import json

AUDIT_LOCK = ROOT / '00_admin' / 'PNADC_HISTORICAL_BACKCAST_HARDENING_AUDIT_LOCK.json'
audit_lock = json.loads(AUDIT_LOCK.read_text(encoding='utf-8'))
print(json.dumps(audit_lock, ensure_ascii=False, indent=2))
assert audit_lock['status'] == 'AUDIT_PASSED', audit_lock['critical_failures']

## 2. Hardening completo

A execução abaixo usa 30 réplicas de bootstrap. Para uma checagem rápida de engenharia, reduza temporariamente para 5. Para o lock final, preserve 30 ou aumente para 100 se houver tempo computacional.

In [ ]:
full = subprocess.run(
    [
        sys.executable, str(SCRIPT),
        '--root', str(ROOT),
        '--mode', 'full',
        '--calibration-folds', '5',
        '--bootstrap-calibration-folds', '3',
        '--bootstrap-reps', '30',
        '--mca-components', '8',
        '--cluster-k-min', '4',
        '--cluster-k-max', '8',
        '--strict',
    ],
    text=True,
    capture_output=True,
    check=False,
)
print(full.stdout)
print(full.stderr)
print('Full exit code:', full.returncode)

In [ ]:
LOCK = ROOT / '00_admin' / 'PNADC_HISTORICAL_BACKCAST_HARDENING_LOCK.json'
lock = json.loads(LOCK.read_text(encoding='utf-8'))
print(json.dumps(lock, ensure_ascii=False, indent=2))
print('STATUS:', lock['status'])
print('Critical failures:', len(lock.get('critical_failures', [])))
print('Warnings:', len(lock.get('warnings', [])))

## 3. Inspeção dos resultados

As células seguintes exibem métricas temporais, golden totals, estimativas finais, suporte e divergência logística × KNN.

In [ ]:
import pandas as pd

artifacts = lock.get('artifacts', {})
for key in [
    'metrics', 'goldens', 'final_estimates', 'support_summary',
    'divergence', 'bootstrap_summary', 'layout'
]:
    print(key, '=>', artifacts.get(key))

In [ ]:
metrics = pd.read_csv(artifacts['metrics'])
metrics

In [ ]:
goldens = pd.read_csv(artifacts['goldens'])
goldens

In [ ]:
final_estimates = pd.read_csv(artifacts['final_estimates'])
final_estimates

In [ ]:
support = pd.read_csv(artifacts['support_summary'])
support

In [ ]:
divergence = pd.read_csv(artifacts['divergence'])
divergence

In [ ]:
if lock['status'] == 'FINAL_CERTIFIED':
    FINAL_LOCK = ROOT / '00_admin' / 'PNADC_HISTORICAL_BACKCAST_FINAL_LOCK.json'
    final_lock = json.loads(FINAL_LOCK.read_text(encoding='utf-8'))
    print(json.dumps(final_lock, ensure_ascii=False, indent=2))
else:
    print('Hardening bloqueado. Examine critical_failures e o relatório antes de qualquer uso substantivo.')